In [1]:
import joblib

In [7]:
from tqdm import tqdm

In [2]:
joblib.load("models/xgb-kfold-binary-13/model.pth")

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.001, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=15, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=150, n_jobs=None,
              num_parallel_tree=None, ...)

In [18]:
import pandas as pd
import numpy as np
import joblib

# Load model and artifacts
model = joblib.load("models/xgb-kfold-binary-13/model.pth")
artifacts = joblib.load("/mnt/object/train/transform_artifacts.pkl")

KEEP_COLS = artifacts["keep_cols"]

CATEGORICAL_ONEHOT = artifacts["categorical_onehot"]
CATEGORICAL_LABEL = artifacts["categorical_label"]
median_values = artifacts["median_values"]
mode_values = artifacts["mode_values"]
label_encoders = artifacts["label_encoders"]
onehot_columns_train = artifacts["onehot_columns"]
scaler = artifacts["scaler"]
numeric_cols = artifacts["numeric_cols"]
LABEL_COL = "risk_level"

# --- Helper functions ---

def parse_txt_to_df(filepath):
    with open(filepath, "r") as f:
        lines = f.readlines()
    data = {}
    for line in lines:
        if ":" in line:
            k, v = line.strip().split(":", 1)
            try:
                data[k.strip()] = float(v.strip())
            except ValueError:
                data[k.strip()] = v.strip()
    return pd.DataFrame([data])

def transform_input_df(df):
    # Load transform artifacts
    # artifacts = joblib.load("/mnt/data/LoanData/train/transform_artifacts.pkl")
    # KEEP_COLS = artifacts["keep_cols"]
    # median_values = artifacts["median_values"]
    # mode_values = artifacts["mode_values"]
    # label_encoders = artifacts["label_encoders"]
    # onehot_columns_train = artifacts["onehot_columns"]
    # scaler = artifacts["scaler"]
    # numeric_cols = artifacts["numeric_cols"]
    # CATEGORICAL_ONEHOT = artifacts["categorical_onehot"]
    # CATEGORICAL_LABEL = artifacts["categorical_label"]

    log_msgs = []

    df = df[KEEP_COLS]
    label_series = df.pop(LABEL_COL).map(lambda x: 0 if x == "Low" else 1)

    # Fill NaNs using training stats
    for col in tqdm(df.columns, desc="Handling NaNs with training stats"):
        if df[col].isnull().sum() > 0:
            if col in mode_values:
                df[col] = df[col].fillna(mode_values[col])
            elif col in median_values:
                df[col] = df[col].fillna(median_values[col])
            else:
                df[col] = df[col].fillna(0)

    # One-hot encoding
    for col in CATEGORICAL_ONEHOT:
        dummies = pd.get_dummies(df[col], prefix=col)
    
        # Filter relevant onehot columns for current variable
        relevant_cols = [c for c in onehot_columns_train if c.startswith(f"{col}_")]
    
            # Add missing dummy columns
        for dummy_col in relevant_cols:
            if dummy_col not in dummies.columns:
                dummies[dummy_col] = 0

        dummies = dummies[relevant_cols]
        df = pd.concat([df.drop(columns=[col]), dummies], axis=1)


    # Label encoding
    for col in CATEGORICAL_LABEL:
        le = label_encoders[col]
        df[col] = le.transform(df[col].astype(str))

    # Standard scaling
    df[numeric_cols] = scaler.transform(df[numeric_cols])
    df[LABEL_COL] = label_series.reset_index(drop=True)

    return df
    

# --- Main execution ---

# Replace with your actual .txt file path
txt_file_path = r"sample_5.txt"

raw_df = parse_txt_to_df(txt_file_path)
print("📦 Raw extracted features:")
display(raw_df)

true_label = transformed_df[LABEL_COL].iloc[0] if LABEL_COL in raw_df.columns else None
# if LABEL_COL in raw_df.columns:
#     raw_df = raw_df.drop(columns=[LABEL_COL])

transformed_df = transform_input_df(raw_df)
if LABEL_COL in transformed_df.columns:
    transformed_df = transformed_df.drop(columns=[LABEL_COL])
print(f"\n🧪 Transformed shape: {transformed_df.shape}")
display(transformed_df)

# Predict
prediction = model.predict(transformed_df.values)[0]
print(f"\n🔮 Predicted risk class: {prediction}")
print(f"✅ True label (if present): {true_label}")


📦 Raw extracted features:


,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,sub_grade,home_ownership,annual_inc,verification_status,dti,...,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,risk_level
0,17000.0,17000.0,17000.0,17.47,426.81,D1,MORTGAGE,80000.0,Source Verified,6.29,...,2.0,83.3,0.0,0.0,0.0,30376.0,10078.0,10300.0,10276.0,Low


Handling NaNs with training stats: 100%|██████████| 61/61 [00:00<00:00, 7434.55it/s]


🧪 Transformed shape: (1, 68)


,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,sub_grade,annual_inc,dti,delinq_2yrs,fico_range_low,...,total_il_high_credit_limit,home_ownership_ANY,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Not Verified,verification_status_Source Verified,verification_status_Verified
0,0.212017,0.21264,0.214529,0.906407,-0.071484,0.749656,0.016062,-0.891612,-0.354305,-1.017377,...,-0.746058,0,True,0,0,0,0,0,True,0



🔮 Predicted risk class: 0
✅ True label (if present): 0


In [9]:
print(transformed_df.columns)

Index(['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate',
       'installment', 'sub_grade', 'annual_inc', 'dti', 'delinq_2yrs',
       'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc',
       'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt',
       'tot_cur_bal', 'open_act_il', 'open_il_12m', 'total_bal_il',
       'open_rv_12m', 'max_bal_bc', 'all_util', 'total_rev_hi_lim', 'inq_fi',
       'total_cu_tl', 'inq_last_12m', 'avg_cur_bal', 'bc_open_to_buy',
       'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mort_acc',
       'mths_since_recent_inq', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
       'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
       'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
       'tax_liens', '

In [10]:
print(raw_df.columns)

Index(['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate',
       'installment', 'sub_grade', 'home_ownership', 'annual_inc',
       'verification_status', 'dti', 'delinq_2yrs', 'fico_range_low',
       'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal',
       'revol_util', 'total_acc', 'collections_12_mths_ex_med',
       'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_act_il',
       'open_il_12m', 'total_bal_il', 'open_rv_12m', 'max_bal_bc', 'all_util',
       'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
       'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths',
       'delinq_amnt', 'mort_acc', 'mths_since_recent_inq', 'num_actv_bc_tl',
       'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl',
       'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats',
       'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m',
       'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75',
    

In [25]:
import pandas as pd
import numpy as np
import joblib

# Load model and artifacts
# model = joblib.load("models/xgb-kfold-binary-13/model.pth")
artifacts = joblib.load("/mnt/object/train/transform_artifacts.pkl")

KEEP_COLS = artifacts["keep_cols"]

CATEGORICAL_ONEHOT = artifacts["categorical_onehot"]
CATEGORICAL_LABEL = artifacts["categorical_label"]
median_values = artifacts["median_values"]
mode_values = artifacts["mode_values"]
label_encoders = artifacts["label_encoders"]
onehot_columns_train = artifacts["onehot_columns"]
scaler = artifacts["scaler"]
numeric_cols = artifacts["numeric_cols"]
LABEL_COL = "risk_level"

# --- Helper functions ---

def parse_txt_to_df(filepath):
    with open(filepath, "r") as f:
        lines = f.readlines()
    data = {}
    for line in lines:
        if ":" in line:
            k, v = line.strip().split(":", 1)
            try:
                data[k.strip()] = float(v.strip())
            except ValueError:
                data[k.strip()] = v.strip()
    return pd.DataFrame([data])

def transform_input_df(df):
    # Load transform artifacts
    # artifacts = joblib.load("/mnt/data/LoanData/train/transform_artifacts.pkl")
    # KEEP_COLS = artifacts["keep_cols"]
    # median_values = artifacts["median_values"]
    # mode_values = artifacts["mode_values"]
    # label_encoders = artifacts["label_encoders"]
    # onehot_columns_train = artifacts["onehot_columns"]
    # scaler = artifacts["scaler"]
    # numeric_cols = artifacts["numeric_cols"]
    # CATEGORICAL_ONEHOT = artifacts["categorical_onehot"]
    # CATEGORICAL_LABEL = artifacts["categorical_label"]

    log_msgs = []

    df = df[KEEP_COLS]
    label_series = df.pop(LABEL_COL).map(lambda x: 0 if x == "Low" else 1)

    # Fill NaNs using training stats
    for col in tqdm(df.columns, desc="Handling NaNs with training stats"):
        if df[col].isnull().sum() > 0:
            if col in mode_values:
                df[col] = df[col].fillna(mode_values[col])
            elif col in median_values:
                df[col] = df[col].fillna(median_values[col])
            else:
                df[col] = df[col].fillna(0)

    # One-hot encoding
    for col in CATEGORICAL_ONEHOT:
        dummies = pd.get_dummies(df[col], prefix=col)
    
        # Filter relevant onehot columns for current variable
        relevant_cols = [c for c in onehot_columns_train if c.startswith(f"{col}_")]
    
            # Add missing dummy columns
        for dummy_col in relevant_cols:
            if dummy_col not in dummies.columns:
                dummies[dummy_col] = 0

        dummies = dummies[relevant_cols]
        df = pd.concat([df.drop(columns=[col]), dummies], axis=1)


    # Label encoding
    for col in CATEGORICAL_LABEL:
        le = label_encoders[col]
        df[col] = le.transform(df[col].astype(str))

    # Standard scaling
    df[numeric_cols] = scaler.transform(df[numeric_cols])
    df[LABEL_COL] = label_series.reset_index(drop=True)

    return df
    

# --- Main execution ---

# Replace with your actual .txt file path
# txt_file_path = r"sample_5.txt"

raw_df = pd.read_csv("/mnt/object/val/val_templatetest.csv")
print("📦 Raw extracted features:")
display(raw_df)

# true_label = transformed_df[LABEL_COL].iloc[0] if LABEL_COL in raw_df.columns else None
# if LABEL_COL in raw_df.columns:
#     raw_df = raw_df.drop(columns=[LABEL_COL])

transformed_df = transform_input_df(raw_df)
if LABEL_COL in transformed_df.columns:
    transformed_df = transformed_df.drop(columns=[LABEL_COL])
print(f"\n🧪 Transformed shape: {transformed_df.shape}")
display(transformed_df)
transformed_df.to_csv("val_templatetest_transformed.csv")
# Predict
# prediction = model.predict(transformed_df.values)[0]
# print(f"\n🔮 Predicted risk class: {prediction}")
# print(f"✅ True label (if present): {true_label}")


📦 Raw extracted features:


,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,...,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,disbursement_method,debt_settlement_flag,risk_level
0,12957010,12000.0,12000.0,12000.0,36 months,15.31,417.81,C,C4,Accounting and Payroll,...,0.0,0.0,20432.0,14286.0,9200.0,6532.0,N,Cash,N,High
1,130935834,19200.0,19200.0,19200.0,60 months,12.61,433.04,C,C1,LPN,...,0.0,0.0,415845.0,93518.0,9500.0,87320.0,N,Cash,N,Low
2,24114692,25000.0,25000.0,25000.0,60 months,8.18,509.07,B,B1,SENIOR ACCOUNTANT,...,0.0,0.0,214497.0,85000.0,14000.0,85450.0,N,Cash,N,Low
3,3644761,7200.0,7200.0,7200.0,36 months,6.62,221.07,A,A2,GE,...,0.0,0.0,361867.0,45271.0,29600.0,7897.0,N,Cash,N,Low
4,126607556,29000.0,29000.0,29000.0,60 months,11.99,644.95,B,B5,Administrative Assistant 2,...,0.0,0.0,154832.0,10521.0,28200.0,14732.0,N,Cash,N,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22602,39700709,19650.0,19650.0,19650.0,36 months,16.49,695.60,D,D3,Registered nurse/Mds coordinator,...,0.0,0.0,61400.0,44358.0,33600.0,20000.0,N,Cash,N,High
22603,39672675,22500.0,22500.0,22500.0,60 months,14.31,527.16,C,C4,senior computer operator,...,0.0,0.0,294130.0,48567.0,28800.0,38630.0,N,Cash,N,Low
22604,107105421,16000.0,16000.0,16000.0,60 months,13.99,372.21,C,C3,Food Service Director,...,1.0,1.0,213077.0,49558.0,6600.0,69644.0,N,Cash,N,High
22605,95140285,24000.0,24000.0,23750.0,36 months,7.99,751.97,A,A5,CPA,...,0.0,0.0,622900.0,52986.0,32500.0,14000.0,N,Cash,N,Low


/tmp/ipykernel_23362/2225538097.py:60: SettingWithCopyWarning: <?, ?it/s]
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna(median_values[col])
Handling NaNs with training stats: 100%|██████████| 61/61 [00:00<00:00, 1820.57it/s]


🧪 Transformed shape: (22607, 68)


,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,sub_grade,annual_inc,dti,delinq_2yrs,fico_range_low,...,total_il_high_credit_limit,home_ownership_ANY,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Not Verified,verification_status_Source Verified,verification_status_Verified
0,-0.331947,-0.331435,-0.329309,0.459341,-0.105163,0.432200,-0.315339,0.176711,-0.354305,-1.168866,...,-0.830362,False,False,0,0,False,True,False,True,False
1,0.451361,0.452033,0.453818,-0.099490,-0.048171,-0.043984,0.347462,0.210163,-0.354305,-0.259933,...,0.988748,False,True,0,0,False,False,True,False,False
2,1.082360,1.083160,1.084670,-1.016388,0.236342,-0.837625,0.098912,-0.031829,0.800804,0.194534,...,0.946641,False,False,0,0,True,False,False,True,False
3,-0.854153,-0.853748,-0.851394,-1.339268,-0.841387,-1.472537,0.405457,-0.636097,-0.354305,0.951978,...,-0.799626,False,True,0,0,False,False,True,False,False
4,1.517531,1.518421,1.519740,-0.227814,0.744821,-0.202713,-0.385761,-0.710118,-0.354305,2.618355,...,-0.645722,False,True,0,0,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22602,0.500318,0.501000,0.502763,0.703571,0.934359,1.067112,0.057487,0.074932,-0.354305,-1.168866,...,-0.527102,False,False,0,0,False,True,False,True,False
22603,0.810378,0.811123,0.812751,0.252367,0.304037,0.432200,-0.108213,0.488454,-0.354305,-0.108444,...,-0.107609,False,True,0,0,False,False,False,True,False
22604,0.103224,0.103825,0.105761,0.186135,-0.275804,0.273472,-0.106722,-0.098021,-0.354305,-1.168866,...,0.590736,False,True,0,0,False,False,True,False,False
22605,0.973567,0.974345,0.948710,-1.055713,1.145302,-0.996353,0.430312,-0.293038,-0.354305,0.043045,...,-0.662205,False,True,0,0,False,False,True,False,False


In [24]:
pwd

'/home/jovyan/work/train'